In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Określanie opcji

<details>
<summary><b>Wersje pakietów</b></summary>

Kod na tej stronie został opracowany przy użyciu poniższych wymagań.
Zalecamy korzystanie z tych lub nowszych wersji.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Możesz używać opcji do dostosowywania prymitywów Estimator i Sampler. Ta sekcja skupia się na tym, jak określać opcje prymitywów Qiskit Runtime. Choć interfejs metody `run()` prymitywów jest wspólny dla wszystkich implementacji, ich opcje już nie. Zapoznaj się z odpowiednimi dokumentacjami API, aby uzyskać informacje o opcjach [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives#primitives) i [`qiskit_aer.primitives`](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html).

Uwagi dotyczące określania opcji w prymitywach:

- `SamplerV2` i `EstimatorV2` mają osobne klasy opcji. Możesz zobaczyć dostępne opcje i aktualizować ich wartości podczas inicjalizacji prymitywu lub po niej.
- Użyj metody `update()`, aby zastosować zmiany do atrybutu `options`.
- Jeśli nie podasz wartości dla opcji, otrzyma ona specjalną wartość `Unset` i zostaną użyte wartości domyślne serwera.
- Atrybut `options` jest typem Python `dataclass`. Możesz użyć wbudowanej metody `asdict`, aby przekonwertować go do słownika.

<span id="pass-options"></span>
## Ustawianie opcji prymitywu
Możesz ustawiać opcje podczas inicjalizacji prymitywu, po jego inicjalizacji lub w metodzie `run()`. Zapoznaj się z sekcją [reguły pierwszeństwa](runtime-options-overview#options-precedence), aby zrozumieć, co się dzieje, gdy ta sama opcja jest podana w wielu miejscach.

### Inicjalizacja prymitywu
Podczas inicjalizacji prymitywu możesz przekazać instancję klasy opcji lub słownik, który następnie tworzy kopię tych opcji. W związku z tym zmiana oryginalnego słownika lub instancji opcji nie wpływa na opcje należące do prymitywów.

#### Klasa opcji
Tworząc instancję klasy `EstimatorV2` lub `SamplerV2`, możesz przekazać instancję klasy opcji. Opcje te zostaną następnie zastosowane podczas używania `run()` do wykonania obliczeń. Podaj opcje w tym formacie: `options.option.sub-option.sub-sub-option = choice`. Na przykład: `options.dynamical_decoupling.enable = True`

Przykład:

`SamplerV2` i `EstimatorV2` mają osobne klasy opcji ([`EstimatorOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) i [`SamplerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options)).

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EstimatorOptions

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

options = EstimatorOptions(
    resilience_level=2,
    resilience={"zne_mitigation": True, "zne": {"noise_factors": [1, 3, 5]}},
)

# or...
options = EstimatorOptions()
options.resilience_level = 2
options.resilience.zne_mitigation = True
options.resilience.zne.noise_factors = [1, 3, 5]

estimator = Estimator(mode=backend, options=options)

#### Słownik
Możesz podać opcje jako słownik podczas inicjalizacji prymitywu.

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(
    backend,
    options={
        "resilience_level": 2,
        "resilience": {
            "zne_mitigation": True,
            "zne": {"noise_factors": [1, 3, 5]},
        },
    },
)

### Aktualizacja opcji po inicjalizacji
Możesz podawać opcje w tym formacie: `primitive.options.option.sub-option.sub-sub-option = choice`, aby skorzystać z automatycznego uzupełniania, lub użyć metody `update()` do zbiorczej aktualizacji.

Klasy opcji `SamplerV2` i `EstimatorV2` ([`EstimatorOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) i [`SamplerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options)) nie muszą być tworzone jako instancje, jeśli ustawiasz opcje po inicjalizacji prymitywu.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

# Setting options after primitive initialization
# This uses auto-complete.
estimator.options.default_shots = 4000
# This does bulk update.
estimator.options.update(
    default_shots=4000, resilience={"zne_mitigation": True}
)

<span id="run-method"></span>
### Metoda Run()
Jedyne wartości, które możesz przekazać do `run()`, to te zdefiniowane w interfejsie. Czyli `shots` dla Samplera i `precision` dla Estimatora. Nadpisuje to każdą wartość ustawioną dla `default_shots` lub `default_precision` w bieżącym uruchomieniu.

In [4]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)

sampler = Sampler(mode=backend)
# Default shots to use if not specified in run()
sampler.options.default_shots = 500
# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

# Sample two circuits with different numbers of shots.
# 100 shots is used for transpiled1 and 200 for transpiled.
sampler.run([(transpiled1, None, 100), (transpiled2, None, 200)])

<RuntimeJobV2('d5k96cn853es738djikg', 'sampler')>

### Special cases

#### Resilience level (Estimator only)

The resilience level is not actually an option that directly impacts the primitive query, but specifies a base set of curated options to build off of. In general, level 0 turns off all error mitigation, level 1 turns on options for measurement error mitigation, and level 2 turns on options for gate and measurement error mitigation.

Any options you manually specify in addition to the resilience level are applied on top of the base set of options defined by the resilience level. Therefore, in principle, you could set the resilience level to 1, but then turn off measurement mitigation, although this is not advised.

In the following example, setting the resilience level to 0 initially turns off `zne_mitigation`, but `estimator.options.resilience.zne_mitigation = True` overrides the relevant setup from `estimator.options.resilience_level = 0`.

In [5]:
from qiskit_ibm_runtime import EstimatorV2, QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = EstimatorV2(backend)

estimator.options.default_shots = 100
estimator.options.resilience_level = 0
estimator.options.resilience.zne_mitigation = True

### Przypadki szczególne
#### Poziom odporności (tylko Estimator)
Poziom odporności nie jest w zasadzie opcją, która bezpośrednio wpływa na zapytanie do prymitywu, ale określa podstawowy zestaw starannie dobranych opcji, na których można budować. Ogólnie rzecz biorąc, poziom 0 wyłącza wszystkie techniki mitygacji błędów, poziom 1 włącza opcje mitygacji błędów pomiarowych, a poziom 2 włącza opcje mitygacji błędów bramek i pomiarów.

Wszelkie opcje, które podasz ręcznie oprócz poziomu odporności, są stosowane na wierzchu podstawowego zestawu opcji zdefiniowanego przez poziom odporności. Dlatego w zasadzie mógłbyś ustawić poziom odporności na 1, ale następnie wyłączyć mitygację pomiarów, choć nie jest to zalecane.

W poniższym przykładzie ustawienie poziomu odporności na 0 wyłącza początkowo `zne_mitigation`, ale `estimator.options.resilience.zne_mitigation = True` nadpisuje odpowiednie ustawienie z `estimator.options.resilience_level = 0`.

In [6]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)


# Setting shots during primitive initialization
sampler = Sampler(mode=backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 2000

# This does bulk update.  The value for default_shots is overridden if you specify shots with run() or in the PUB.
sampler.options.update(
    default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"}
)

# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d5k96icjt3vs73ds5t0g', 'sampler')>

#### Strzały (tylko Sampler)
Metoda `SamplerV2.run` przyjmuje dwa argumenty: listę PUB-ów, z których każdy może określać wartość strzałów specyficzną dla danego PUB, oraz argument kluczowy shots. Te wartości strzałów są częścią interfejsu wykonawczego Samplera i są niezależne od opcji Samplera Runtime. Mają pierwszeństwo przed wszelkimi wartościami określonymi jako opcje, aby zachować zgodność z abstrakcją Samplera.

Jednak jeśli `shots` nie jest określone przez żaden PUB ani w argumencie kluczowym run (lub jeśli wszystkie są `None`), to używana jest wartość strzałów z opcji, w szczególności `default_shots`.

Podsumowując, oto kolejność pierwszeństwa przy określaniu strzałów w Samplerze, dla dowolnego PUB:

1. Jeśli PUB określa strzały, użyj tej wartości.
2. Jeśli argument kluczowy `shots` jest podany w `run`, użyj tej wartości.
3. Jeśli `num_randomizations` i `shots_per_randomization` są podane jako opcje `twirling`, strzały są iloczynem tych wartości.
3. Jeśli `sampler.options.default_shots` jest podane, użyj tej wartości.

Zatem jeśli strzały są podane we wszystkich możliwych miejscach, używane jest to z najwyższym priorytetem (strzały podane w PUB).

#### Precyzja (tylko Estimator)
Precyzja jest analogiczna do strzałów opisanych w poprzedniej sekcji, z tą różnicą, że opcje Estimatora zawierają zarówno `default_shots`, jak i `default_precision`. Ponadto, ponieważ twirling bramek jest domyślnie włączony, iloczyn `num_randomizations` i `shots_per_randomization` ma pierwszeństwo przed tymi dwiema opcjami.

W szczególności, dla dowolnego PUB Estimatora:

1. Jeśli PUB określa precyzję, użyj tej wartości.
2. Jeśli argument kluczowy precision jest podany w `run`, użyj tej wartości.
2. Jeśli `num_randomizations` i `shots_per_randomization` są podane jako [opcje `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options) (domyślnie włączone), użyj ich iloczynu do kontrolowania ilości danych.
3. Jeśli `estimator.options.default_shots` jest podane, użyj tej wartości do kontrolowania ilości danych.
4. Jeśli `estimator.options.default_precision` jest podane, użyj tej wartości.

Na przykład jeśli precyzja jest podana we wszystkich czterech miejscach, używane jest to z najwyższym priorytetem (precyzja podana w PUB).

> **Note:** Precyzja skaluje się odwrotnie proporcjonalnie do użycia. To znaczy im niższa precyzja, tym więcej czasu QPU potrzeba do uruchomienia.

## Często używane opcje
Dostępnych jest wiele opcji, ale poniżej przedstawiono te najczęściej używane:

<span id="shots"></span>
### Strzały
W przypadku niektórych algorytmów ustawienie określonej liczby strzałów jest kluczową częścią ich procedur. Strzały (lub precyzja) można podawać w wielu miejscach. Priorytetyzowane są w następujący sposób:

Dla dowolnego PUB Samplera:

1. Całkowitoliczbowe strzały zawarte w PUB
2. Wartość `run(...,shots=val)`
3. Wartość `options.default_shots`

Dla dowolnego PUB Estimatora:

1. Zmiennoprzecinkowa precyzja zawarta w PUB
2. Wartość `run(...,precision=val)`
3. Wartość `options.default_shots`
4. Wartość `options.default_precision`

Przykład:

In [7]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

estimator.options.max_execution_time = 2500

<span id="no-error-mitigation"></span>
## Turn off all error mitigation and error suppression

You can turn off all error mitigation and suppression if you are, for example, doing research on your own mitigation techniques. To accomplish this, for EstimatorV2, set `resilience_level = 0`. For SamplerV2, no changes are necessary because no error mitigation or suppression options are enabled by default.

Example:

Turn off all error mitigation and suppression in Estimator.

In [8]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService

# Define the service.  This allows you to access IBM QPU.
service = QiskitRuntimeService()

# Get a backend
backend = service.least_busy(operational=True, simulator=False)

# Define Estimator
estimator = Estimator(backend)

options = estimator.options

# Turn off all error mitigation and suppression
options.resilience_level = 0

### Maksymalny czas wykonania
Maksymalny czas wykonania (`max_execution_time`) ogranicza czas działania zadania. Jeśli zadanie przekroczy ten limit czasu, zostanie przymusowo anulowane. Ta wartość dotyczy pojedynczych zadań, niezależnie od tego, czy są uruchamiane w trybie zadania, sesji czy wsadowym.

Wartość jest ustawiana w sekundach na podstawie czasu kwantowego (nie czasu zegarowego), czyli czasu, przez który QPU jest dedykowany przetwarzaniu twojego zadania. Jest ignorowana podczas korzystania z trybu lokalnego testowania, ponieważ ten tryb nie używa czasu kwantowego.